# 요리 종류 열 추가 (master.parquet)
순서: 1 → 2 → 3(검수, 필요하면 OVERRIDE 수정 후 재실행) → 4(저장). 저장 전에 원본은 `_백업_요리종류전.parquet`으로 복사해 둬요.

In [ ]:
# 1) 준비: 드라이브 연결, master.parquet 읽기
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json, time, shutil
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
MASTER = '/content/drive/MyDrive/혈당코치/mapping2/master.parquet'
CACHE  = '/content/drive/MyDrive/혈당코치/mapping2/cuisine_cache.json'

M = pd.read_parquet(MASTER)
print(len(M), '개 음식 / 열:', list(M.columns)[:12])
M[['food_id', 'ai_hub_name', '식품명', '식품대분류', '식품상세분류']].head()

In [ ]:
# 2) LLM으로 요리 종류 태깅 (40개씩 묶어서 호출, 결과는 캐시에 저장)
CUISINES = ['한식', '중식', '일식', '양식', '분식', '기타']
PROMPT = """아래 음식들이 어느 요리 종류인지 분류해 주세요. 한국 식당에서 파는 기준으로 판단해요.
- 한식: 밥, 국, 찌개, 구이, 나물, 전, 김치 등 전통 한국 음식
- 중식: 짜장면, 짬뽕, 탕수육, 마파두부, 볶음밥(중화풍) 등 중화요리
- 일식: 초밥, 돈까스, 우동, 라멘, 덮밥(일본식) 등
- 양식: 피자, 파스타, 스테이크, 샐러드, 샌드위치, 햄버거 등
- 분식: 떡볶이, 김밥, 순대, 라면, 튀김, 어묵 등
- 기타: 위에 해당하지 않거나 애매한 것(디저트, 음료, 동남아 음식 등)
애매하면 '기타'로 하세요. 추측해서 한식으로 넣지 마세요.

{items}

JSON으로만 답하세요: {{"결과": {{"음식id": "요리종류", ...}}}}"""

cache = json.load(open(CACHE, encoding='utf-8')) if os.path.exists(CACHE) else {}

def tag_batch(rows):
    items = '\n'.join(f"{r.food_id}: {r.ai_hub_name} (분류: {r.식품대분류}/{r.식품상세분류})" for r in rows.itertuples())
    for _ in range(3):
        try:
            resp = client.chat.completions.create(model=MODEL, temperature=0, response_format={'type': 'json_object'},
                                                  messages=[{'role': 'user', 'content': PROMPT.format(items=items)}])
            out = json.loads(resp.choices[0].message.content)['결과']
            return {str(k): (v if v in CUISINES else '기타') for k, v in out.items()}
        except Exception as e:
            print('재시도:', type(e).__name__, e); time.sleep(2)
    return {}

todo = M[~M['food_id'].astype(str).isin(cache)]
for s in range(0, len(todo), 40):
    cache.update(tag_batch(todo.iloc[s:s + 40]))
    json.dump(cache, open(CACHE, 'w', encoding='utf-8'), ensure_ascii=False)
    print(f'{min(s + 40, len(todo))}/{len(todo)}')

M['요리종류'] = M['food_id'].astype(str).map(cache).fillna('기타')
print(M['요리종류'].value_counts())

In [ ]:
# 3) 검수: 중식·일식을 눈으로 훑어보고, 틀린 건 OVERRIDE에 적어서 고쳐요
OVERRIDE = {
    # '음식 이름': '중식',   ← 예시. 틀린 것만 이렇게 적고 이 셀을 다시 실행
}
for name, c in OVERRIDE.items():
    M.loc[M['ai_hub_name'] == name, '요리종류'] = c

for c in ['중식', '일식', '양식', '분식']:
    print(f'\n[{c}] {(M["요리종류"] == c).sum()}개')
    print(', '.join(M.loc[M['요리종류'] == c, 'ai_hub_name']))
print('\n[기타]'); print(', '.join(M.loc[M['요리종류'] == '기타', 'ai_hub_name']))

In [ ]:
# 4) 저장: 원본은 백업해 두고 요리종류 열이 들어간 master.parquet으로 덮어쓰기
BACKUP = MASTER.replace('.parquet', '_백업_요리종류전.parquet')
if not os.path.exists(BACKUP):
    shutil.copy(MASTER, BACKUP)
M.to_parquet(MASTER, index=False)

# 확인: 저장된 파일 다시 읽기
chk = pd.read_parquet(MASTER)
print('저장 완료:', len(chk), '개 /', '요리종류' in chk.columns)
print(chk['요리종류'].value_counts().to_dict())